In [1]:
import math
import torch
import pathlib
import torchvision
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torchvision.transforms.v2

In [2]:
import dataset as datasets
import model as models

In [3]:
dataframe_train, dataframe_test = datasets.make_train_test_split(pathlib.Path("/opt/datasets/imagenet-256-dimensi0n/"))

dataset_train = datasets.ImageNetDataset(dataframe=dataframe_train, transform=datasets.transform_extra)
dataset_test = datasets.ImageNetDataset(dataframe=dataframe_test, transform=datasets.transform_basic)

dataloader_train = torch.utils.data.DataLoader(
    dataset=dataset_train,
    batch_size=64+32+4,
    shuffle=True,
    num_workers=8
)

dataloader_test = torch.utils.data.DataLoader(
    dataset=dataset_test,
    batch_size=512,
    shuffle=True,
    num_workers=8
)

len(dataset_train), len(dataset_test), len(dataloader_train), len(dataloader_test)

(404869, 134957, 4049, 264)

In [4]:
X_batch, y_batch = next(iter(dataloader_train))
X_batch.shape, y_batch.shape

(torch.Size([100, 3, 256, 256]), torch.Size([100]))

In [5]:
model = models.LightViTClassifier(
    target_resolution=(256, 256),
    num_classes=len(dataset_train.class_to_idx),
    num_patches=(16, 16),
    embedding_dim=512,
    num_layers=16,
    num_heads=8,
    dropout_p=0.1,
    in_channels=3
)

param_count = sum([param.numel() for param in model.parameters()])
print(f"* LightViTClassifier parameter count: {param_count:,d}")

* LightViTClassifier parameter count: 54,063,592


In [6]:
class_counts = dataframe_train["label"].value_counts().sort_index()
class_weights = class_counts.sum() / (len(class_counts) * torch.tensor(class_counts.values, dtype=torch.float32))
class_weights[:8]

tensor([0.8633, 1.3145, 0.9223, 1.0096, 0.6326, 0.6616, 1.2732, 1.0739])

In [7]:
def init_scheduler(num_epochs, steps_per_epoch, num_warmup_epochs):
    total_steps = num_epochs * steps_per_epoch
    warmup_steps = num_warmup_epochs * steps_per_epoch

    def scheduler(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        cosine = 0.5 * (1 + math.cos(math.pi * progress))
        return 0.01 + (1 - 0.01) * cosine

    return scheduler

In [8]:
def checkpoint(model, optimizer, history, epoch, session=1, name="LightViT54M", save_dir="./checkpoints"):
    path = f"{save_dir}/{name}-T{session}-E{epoch:0>3}-{{}}.pth"
    torch.save(model.state_dict(), path.format("model"))
    torch.save(optimizer.state_dict(), path.format("optimizer"))
    torch.save(history, path.format("history"))

In [9]:
def predict(model, dataloader, max_batch=None, device="cuda"):
    model.eval()
    torch.cuda.empty_cache()
    y_pred, y_true = [], []
    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader, 1):
            y_logits = model(X.to(device))
            y_pred += y_logits.argmax(dim=1).tolist()
            y_true += y.tolist()
            print(f"* Evaluating {batch}/{max_batch or len(dataloader)}", end="\r")
            if batch >= max_batch: break
    return torch.tensor(y_pred), torch.tensor(y_true)

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.LightViTClassifier(
    target_resolution=(256, 256),
    num_classes=len(dataset_train.class_to_idx),
    num_patches=(16, 16),
    embedding_dim=512,
    num_layers=16,
    num_heads=8,
    dropout_p=0.1,
    in_channels=3
).to(device)

optimizer = torch.optim.AdamW(
    params=model.parameters(),
    lr=2e-4,
    betas=(0.9, 0.999),
    eps=1e-6,
    weight_decay=0.01
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device),
    label_smoothing=0.1
)

num_epochs = 128

accumulation_steps = 5

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=init_scheduler(
        num_epochs=num_epochs,
        steps_per_epoch=len(dataloader_train) // accumulation_steps,
        num_warmup_epochs=10
    )
)

history = {
    "raw_batch_loss": [],
    "avg_batch_loss": [],
    "avg_batch_accuracy": [],
    "epoch_accuracy": [],
    "learning_rate": []
}

In [11]:
y_pred, y_true = predict(model=model, dataloader=dataloader_test, max_batch=5, device=device)
baseline_accuracy = (y_pred == y_true).sum() / len(y_true)
baseline_accuracy*100, y_pred[:5], y_true[:5]

(tensor(0.0391),
 tensor([985, 193, 193, 193, 193]),
 tensor([381, 362,  11, 548, 408]))

In [ ]:
for epoch in range(1, num_epochs+1):
    model.train()
    optimizer.zero_grad()

    prediction_matches, prediction_count = 0, 0
    for batch, (X, y) in enumerate(dataloader_train, 1):
        X = X.to(device)
        y = y.to(device)

        with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
            y_logits = model(X)
            loss = criterion(y_logits, y) / accumulation_steps

        loss.backward()

        if batch % accumulation_steps == 0:
            nn.utils.clip_grad_norm_(parameters=model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step()

        loss_batch = loss.item() * accumulation_steps
        history["raw_batch_loss"].append(loss_batch)
        loss_avg = sum(history["raw_batch_loss"][-500:]) / len(history["raw_batch_loss"][-500:])
        history["avg_batch_loss"].append(loss_avg)

        prediction_matches += (y_logits.argmax(dim=1) == y).sum().item()
        prediction_count += y.shape[0]
        accuracy_avg = prediction_matches / prediction_count
        history["avg_batch_accuracy"].append(accuracy_avg)

        learning_rate = scheduler.get_last_lr()[0]
        history["learning_rate"].append(learning_rate)
        
        if batch == 1 or batch % accumulation_steps == 0 or batch == len(dataloader_train):
            print(
                f"Epoch [{epoch:0>3}/{num_epochs}] Batch [{batch:0>4}/{len(dataloader_train)}]",
                f"Loss: {loss_batch:.2f} {loss_avg:.2f},",
                f"Accuracy: {accuracy_avg*100:.2f}%,",
                f"LR: {learning_rate:.8f}",
                sep=" ",
                end="\r"
            )
        
        if batch == 1 or batch % 500 == 0 or batch == len(dataloader_train):
            checkpoint(model, optimizer, history, epoch)
            print(end="\n")
    
    history["epoch_accuracy"].append(prediction_matches / prediction_count)

Epoch [001/128] Batch [0001/4049] Loss: 7.02 7.02, Accuracy: 0.00%, LR: 0.00000002
Epoch [001/128] Batch [0500/4049] Loss: 6.97 6.97, Accuracy: 0.10%, LR: 0.00000250
Epoch [001/128] Batch [1000/4049] Loss: 6.96 6.96, Accuracy: 0.11%, LR: 0.00000497
Epoch [001/128] Batch [1500/4049] Loss: 6.94 6.95, Accuracy: 0.13%, LR: 0.00000744
Epoch [001/128] Batch [2000/4049] Loss: 6.92 6.93, Accuracy: 0.18%, LR: 0.00000991
Epoch [001/128] Batch [2500/4049] Loss: 6.89 6.90, Accuracy: 0.22%, LR: 0.00001239
Epoch [001/128] Batch [3000/4049] Loss: 6.82 6.87, Accuracy: 0.28%, LR: 0.00001486
Epoch [001/128] Batch [3500/4049] Loss: 6.78 6.83, Accuracy: 0.33%, LR: 0.00001733
Epoch [001/128] Batch [4000/4049] Loss: 6.74 6.80, Accuracy: 0.37%, LR: 0.00001980
Epoch [001/128] Batch [4049/4049] Loss: 6.73 6.80, Accuracy: 0.37%, LR: 0.00002002
Epoch [002/128] Batch [0001/4049] Loss: 6.87 6.80, Accuracy: 0.00%, LR: 0.00002002
Epoch [002/128] Batch [0500/4049] Loss: 6.79 6.77, Accuracy: 0.67%, LR: 0.00002250
Epoc